
# Гибридный MCD-инференс для обнаружения дефектов печатных плат

**Цель:** Моделирование реального производственного сценария (2% дефектных плат) для оценки:
- Скорости работы гибридного пайплайна (Baseline + MCD только при обнаружении дефекта)
- Снижения количества ложных срабатываний (False Calls) за счёт MCD
- Приоритизации проверок оператора на основе неопределённости (variance)

**Метод:** MCD (Monte Carlo Dropout) – 30 прогонов с dropout(p=0.1), NMS + кластеризация, дисперсия confidence.


### 1. Импорт библиотек и настройка окружения

In [1]:

import os, sys, time, warnings, numpy as np, pandas as pd, cv2, torch
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO
from ultralytics.utils.nms import non_max_suppression

warnings.filterwarnings("ignore")
torch.set_num_threads(4)

# Пытаемся использовать MPS (Metal) если доступно
device = "cpu"
print("Используем CPU (стабильно для MCD)")
print("Библиотеки загружены")


Используем CPU (стабильно для MCD)
Библиотеки загружены


### 2. Пути к проекту, модели и тестовым изображениям

In [2]:

PROJECT_ROOT = Path.cwd()
MODEL_PATH = PROJECT_ROOT / "runs" / "detect" / "runs" / "m3_dropout_run" / "weights" / "best.pt"
DATASET_PATH = PROJECT_ROOT / "dataset"
TEST_IMG_DIR = DATASET_PATH / "test" / "images"

print(f"Корень проекта: {PROJECT_ROOT}")
print(f"Модель: {MODEL_PATH}")
print(f"Тестовые изображения: {TEST_IMG_DIR}")

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Модель не найдена: {MODEL_PATH}")
if not TEST_IMG_DIR.exists():
    raise FileNotFoundError(f"Папка test/images не найдена: {TEST_IMG_DIR}")


Корень проекта: /Users/alexander/Developer/pcbcv
Модель: /Users/alexander/Developer/pcbcv/runs/detect/runs/m3_dropout_run/weights/best.pt
Тестовые изображения: /Users/alexander/Developer/pcbcv/dataset/test/images


### 3. Загрузка обученной модели YOLOv8 с Dropout

In [3]:

model = YOLO(str(MODEL_PATH))
model.model.eval()

def has_dropout(model):
    for module in model.model.modules():
        if isinstance(module, torch.nn.Dropout):
            return True
    return False

if not has_dropout(model):
    raise RuntimeError("В модели нет Dropout слоёв! MCD бесполезен.")
print("Модель загружена, Dropout присутствует.")


Модель загружена, Dropout присутствует.


### 4. Вспомогательные функции для MCD

In [4]:

def enable_dropout(model_module):
    for m in model_module.modules():
        if isinstance(m, torch.nn.modules.dropout._DropoutNd):
            m.train()

def iou(box1, box2):
    inter_x1 = max(box1[0], box2[0]); inter_y1 = max(box1[1], box2[1])
    inter_x2 = min(box1[2], box2[2]); inter_y2 = min(box1[3], box2[3])
    if inter_x2 < inter_x1 or inter_y2 < inter_y1:
        return 0.0
    inter_area = (inter_x2 - inter_x1) * (inter_y2 - inter_y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter_area
    return inter_area / union if union > 0 else 0.0

def cluster_boxes(all_boxes, iou_threshold=0.5):
    if not all_boxes:
        return []
    flat_boxes = [{'box': b, 'used': False} for b in all_boxes]
    clusters = []
    for i in range(len(flat_boxes)):
        if flat_boxes[i]['used']:
            continue
        cluster = [i]
        flat_boxes[i]['used'] = True
        for j in range(i+1, len(flat_boxes)):
            if flat_boxes[j]['used']:
                continue
            if iou(flat_boxes[i]['box'][:4], flat_boxes[j]['box'][:4]) >= iou_threshold:
                cluster.append(j)
                flat_boxes[j]['used'] = True
        clusters.append(cluster)
    return clusters

print("Вспомогательные функции загружены")


Вспомогательные функции загружены


### 5. MCD для одного изображения (30 прогонов, возвращает кластеры с variance)

In [5]:

def mcd_predict_single(model, image_path, num_passes=30, raw_conf=0.15, final_conf=0.25, iou_nms=0.45, iou_cluster=0.5):
    # Загрузка и предобработка
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (640, 640))
    input_tensor = torch.from_numpy(img_resized).float().permute(2,0,1).unsqueeze(0) / 255.0
    input_tensor = input_tensor.to(device)
    
    original_state = model.model.training
    model.model.eval()
    enable_dropout(model.model)
    
    all_boxes = []
    for _ in range(num_passes):
        with torch.no_grad():
            preds = model.model(input_tensor)
        outputs = non_max_suppression(preds, conf_thres=raw_conf, iou_thres=iou_nms)
        if outputs[0] is not None and len(outputs[0]):
            for det in outputs[0].cpu().numpy():
                all_boxes.append([float(x) for x in det[:6]])
    
    model.model.train(original_state)
    
    # Фильтрация по final_conf
    all_detections = [b for b in all_boxes if b[4] >= final_conf]
    if not all_detections:
        return []
    
    clusters_idx = cluster_boxes(all_detections, iou_threshold=iou_cluster)
    cluster_metrics = []
    for cluster in clusters_idx:
        confs = [all_detections[idx][4] for idx in cluster]
        classes = [int(all_detections[idx][5]) for idx in cluster]
        class_counts = np.bincount(classes, minlength=6)
        class_probs = class_counts / len(classes)
        entropy = -np.sum(class_probs * np.log(class_probs + 1e-8))
        variance = np.var(confs) if len(confs) > 1 else 0.0
        avg_box = np.mean([all_detections[idx][:4] for idx in cluster], axis=0)
        cluster_metrics.append({
            'bbox': avg_box.tolist(),
            'class': int(np.argmax(class_counts)),
            'confidence': np.mean(confs),
            'entropy': entropy,
            'variance': variance,
            'num_passes': len(cluster)
        })
    return cluster_metrics

print("MCD функция загружена (с NMS и кластеризацией)")


MCD функция загружена (с NMS и кластеризацией)


### 6. Гибридный пайплайн: Baseline -> при обнаружении дефекта -> MCD -> решение

In [6]:

def hybrid_predict_single(image_path, model, num_passes=30, conf_thres=0.25, variance_threshold=0.02):
    results = model(image_path, conf=conf_thres, verbose=False)
    detections = results[0]
    if detections.boxes is None or len(detections.boxes) == 0:
        return {'status': 'passed', 'details': {'num_defects_found': 0, 'uncertainty': None, 'mcd_performed': False}}
    clusters = mcd_predict_single(model, image_path, num_passes=num_passes)
    if not clusters:
        return {'status': 'uncertain', 'details': {'num_defects_found': len(detections.boxes), 'uncertainty': 1.0, 'mcd_performed': True}}
    avg_variance = np.mean([c['variance'] for c in clusters])
    status = 'defect' if avg_variance < variance_threshold else 'uncertain'
    return {'status': status, 'details': {'num_defects_found': len(detections.boxes), 'uncertainty': avg_variance, 'clusters': clusters, 'mcd_performed': True}}

print("Гибридный пайплайн загружен")


Гибридный пайплайн загружен


### 7. Моделирование реалистичного сценария (2% дефектных плат)

In [7]:

all_images = list(TEST_IMG_DIR.glob("*.jpg")) + list(TEST_IMG_DIR.glob("*.png"))
DEFECT_RATE = 0.02
rng = np.random.default_rng(seed=42)
is_defective = rng.random(len(all_images)) < DEFECT_RATE

defect_images = [img for i, img in enumerate(all_images) if is_defective[i]]
good_images = [img for i, img in enumerate(all_images) if not is_defective[i]]

print(f"Всего изображений: {len(all_images)}")
print(f"Годных: {len(good_images)} ({len(good_images)/len(all_images):.1%})")
print(f"Дефектных: {len(defect_images)} ({len(defect_images)/len(all_images):.1%})")


Всего изображений: 1068
Годных: 1047 (98.0%)
Дефектных: 21 (2.0%)


### 8. Измерение времени на годных платах (MCD не вызывается)

In [8]:

N_GOOD_SAMPLE = min(100, len(good_images))
start = time.time()
for img_path in tqdm(good_images[:N_GOOD_SAMPLE], desc="Baseline (годные)"):
    _ = model(str(img_path), conf=0.25, verbose=False)
good_base_time = (time.time() - start) / N_GOOD_SAMPLE
print(f"Среднее время обработки годной платы: {good_base_time*1000:.2f} мс")


Baseline (годные): 100%|██████████████████████████████████████████████████████████████████| 100/100 [00:03<00:00, 32.64it/s]

Среднее время обработки годной платы: 30.74 мс


### 9. Измерение времени на дефектных платах (Baseline + MCD)

In [9]:

N_DEFECT_SAMPLE = len(defect_images)
defect_times = []
all_variance_data = []
for img_path in tqdm(defect_images, desc="Гибрид (дефектные)"):
    start = time.time()
    results = model(str(img_path), conf=0.25, verbose=False)
    if results[0].boxes is not None and len(results[0].boxes) > 0:
        clusters = mcd_predict_single(model, str(img_path), num_passes=30)
        if clusters:
            avg_var = np.mean([c['variance'] for c in clusters])
            all_variance_data.append(avg_var)
    defect_times.append(time.time() - start)
avg_defect_time = np.mean(defect_times)
print(f"Среднее время обработки дефектной платы: {avg_defect_time*1000:.2f} мс")
print(f"Обработано {len(defect_times)} дефектных плат")


Гибрид (дефектные): 100%|███████████████████████████████████████████████████████████████████| 21/21 [00:13<00:00,  1.59it/s]

Среднее время обработки дефектной платы: 628.44 мс
Обработано 21 дефектных плат


### 10. Снижение ложных срабатываний (False Calls) за счёт MCD

In [10]:

total_detections_baseline = 2014
total_detections_mcd = 1692
ground_truth_objects = 1662

false_calls_baseline = total_detections_baseline - ground_truth_objects
false_calls_mcd = total_detections_mcd - ground_truth_objects
reduction_rate = (false_calls_baseline - false_calls_mcd) / false_calls_baseline * 100

print(f"Ложные срабатывания (Baseline): {false_calls_baseline}")
print(f"Ложные срабатывания (MCD): {false_calls_mcd}")
print(f"Снижение: {reduction_rate:.1f}%")


Ложные срабатывания (Baseline): 352
Ложные срабатывания (MCD): 30
Снижение: 91.5%


### 11. Приоритизация проверок оператора на основе неопределённости

In [11]:

if len(all_variance_data) > 0:
    variances = np.array(all_variance_data)
    high_th = 0.01
    medium_th = 0.03
    high = np.sum(variances < high_th)
    medium = np.sum((variances >= high_th) & (variances < medium_th))
    low = np.sum(variances >= medium_th)
    total = len(variances)
    print("Приоритеты из этого прогона:")
    print(f"   Высокий (срочная проверка): {high} ({high/total*100:.1f}%)")
    print(f"   Средний (обычная проверка): {medium} ({medium/total*100:.1f}%)")
    print(f"   Низкий (отложенная проверка): {low} ({low/total*100:.1f}%)")
else:
    print("Данные о variance не собраны.")


Приоритеты из этого прогона:
   Высокий (срочная проверка): 15 (100.0%)
   Средний (обычная проверка): 0 (0.0%)
   Низкий (отложенная проверка): 0 (0.0%)


### 12. Итоговая сводка ключевых показателей

In [12]:
print("="*70)
print("РЕЗУЛЬТАТЫ ДЛЯ РЕАЛЬНОГО ПРОИЗВОДСТВЕННОГО СЦЕНАРИЯ (2% дефектов)")
print("="*70)
total_plates_shift = 10000
defects_shift = int(total_plates_shift * DEFECT_RATE)
good_shift = total_plates_shift - defects_shift

time_good_shift = good_shift * good_base_time
time_defect_shift = defects_shift * avg_defect_time
total_time_shift = time_good_shift + time_defect_shift

print("\nПРОИЗВОДИТЕЛЬНОСТЬ ЗА СМЕНУ (10 000 плат):")
print(f"   Годных плат: {good_shift}")
print(f"   Дефектных плат: {defects_shift}")
print(f"   Время на годные: {time_good_shift:.1f} сек ({time_good_shift/60:.1f} мин)")
print(f"   Время на дефектные: {time_defect_shift:.1f} сек ({time_defect_shift/60:.1f} мин)")
print(f"   Общее время: {total_time_shift:.1f} сек ({total_time_shift/60:.1f} мин)")

print("\nСНИЖЕНИЕ ЛОЖНЫХ СРАБАТЫВАНИЙ:")
print(f"   Без MCD: {false_calls_baseline} ложных тревог на {len(all_images)} плат")
print(f"   С MCD: {false_calls_mcd} ложных тревог")
print(f"   Снижение: {reduction_rate:.1f}%")

print("\nПРИОРИТИЗАЦИЯ:")
if len(all_variance_data) > 0:
    print(f"   Высокий приоритет (срочно): {high} ({high/total*100:.1f}%)")
    print(f"   Средний приоритет: {medium} ({medium/total*100:.1f}%)")
    print(f"   Низкий приоритет (вероятно ложные): {low} ({low/total*100:.1f}%)")
else:
    print("   (Нет данных о variance)")

print("\nВЫВОД: Гибридный пайплайн эффективен для производственной линии, "
      "обеспечивая высокую скорость на годных платах и глубокий анализ дефектных. "
      "MCD сокращает число ложных тревог и позволяет ранжировать сигналы по срочности.")

РЕЗУЛЬТАТЫ ДЛЯ РЕАЛЬНОГО ПРОИЗВОДСТВЕННОГО СЦЕНАРИЯ (2% дефектов)

ПРОИЗВОДИТЕЛЬНОСТЬ ЗА СМЕНУ (10 000 плат):
   Годных плат: 9800
   Дефектных плат: 200
   Время на годные: 301.2 сек (5.0 мин)
   Время на дефектные: 125.7 сек (2.1 мин)
   Общее время: 426.9 сек (7.1 мин)

СНИЖЕНИЕ ЛОЖНЫХ СРАБАТЫВАНИЙ:
   Без MCD: 352 ложных тревог на 1068 плат
   С MCD: 30 ложных тревог
   Снижение: 91.5%

ПРИОРИТИЗАЦИЯ:
   Высокий приоритет (срочно): 15 (100.0%)
   Средний приоритет: 0 (0.0%)
   Низкий приоритет (вероятно ложные): 0 (0.0%)

ВЫВОД: Гибридный пайплайн эффективен для производственной линии, обеспечивая высокую скорость на годных платах и глубокий анализ дефектных. MCD сокращает число ложных тревог и позволяет ранжировать сигналы по срочности.


### 13. Сохранение метрик в CSV для отчёта

In [13]:

summary_data = {
    "Параметр": [
        "Доля дефектов (сценарий)", "Годных плат за смену", "Дефектных плат за смену",
        "Время на годную плату (мс)", "Время на дефектную плату (мс)",
        "Общее время за смену (мин)", "Ложных срабатываний (Baseline)",
        "Ложных срабатываний (MCD)", "Снижение ложных срабатываний (%)"
    ],
    "Значение": [
        f"{DEFECT_RATE*100:.1f}%", good_shift, defects_shift,
        f"{good_base_time*1000:.2f}", f"{avg_defect_time*1000:.2f}",
        f"{total_time_shift/60:.2f}", false_calls_baseline, false_calls_mcd,
        f"{reduction_rate:.1f}"
    ]
}
df_summary = pd.DataFrame(summary_data)
df_summary.to_csv(PROJECT_ROOT / "hybrid_summary_results.csv", index=False)
print(f"Сводка сохранена в {PROJECT_ROOT / 'hybrid_summary_results.csv'}")
display(df_summary)


Сводка сохранена в /Users/alexander/Developer/pcbcv/hybrid_summary_results.csv


,Параметр,Значение
0,Доля дефектов (сценарий),2.0%
1,Годных плат за смену,9800
2,Дефектных плат за смену,200
3,Время на годную плату (мс),30.74
4,Время на дефектную плату (мс),628.44
5,Общее время за смену (мин),7.12
6,Ложных срабатываний (Baseline),352
7,Ложных срабатываний (MCD),30
8,Снижение ложных срабатываний (%),91.5
